# Data Agent Deployment Notebook

This Notebook demonstrates how to build and deploy a Fabric Data Agent using the Fabric Data Agent SDK as part of an automated CI/CD approach.

Instead of manually configuring a Data Agent through the UI, this Notebook defines the full Data Agent in code and generates it programmatically in the target environment, including instructions, data sources, and configuration settings.

The Notebook is designed to be the deployable unit in your application lifecycle. When executed in a specific environment (Development, Test, or Production), it creates a new Data Agent instance directly in that workspace and binds it to the local Fabric items available in that environment.

This makes the deployment process environment-aware by design. Rather than moving a pre-configured Data Agent across environments and fixing broken references afterwards, the Data Agent is recreated in each stage using the same definition, but with environment-specific resolution of resources such as Lakehouses, Warehouses, and Semantic Models.

During execution, the Notebook uses the SDK to programmatically define the Data Agent structure, apply instructions (written in Markdown), configure behavioral settings, and attach one or more data sources. These data sources are resolved based on the workspace context in which the Notebook runs, ensuring that Development connects to Development assets, Test to Test, and Production to Production.

This approach aligns with infrastructure-as-code principles and improves consistency across environments. It reduces manual configuration steps, eliminates fragile post-deployment reconfiguration, and ensures that each environment is always created from a known and controlled definition.

In addition, this pattern integrates naturally with CI/CD pipelines. The Notebook can be promoted through Git-based workflows or deployment pipelines, and executed as part of the release process to fully materialize the Data Agent in each target stage.

For full detail and voice over of the approach and functioning of this Notebook, please see [this blog](https://data-marc.com/2026/06/23/rethinking-fabric-data-agent-deployment-using-the-data-agent-sdk/)

![Blog header](https://data-marc.com/wp-content/uploads/2026/06/rethinkfabricdataagentdeployment_featuredimage.png)


### Install Fabric Data Agent SDK

In [2]:
# Install Data Agent SDK
%pip install -U fabric-data-agent-sdk --q

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 9, Finished, Available, Finished, False)

Reason for being yanked: Yanked due to conflicts with CVE-2024-35195 mitigation
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
)

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 11, Finished, Available, Finished, False)

## Create the Data Agent
This section covers two tasks:
- Create a new Fabric Data Agent
- Add the Agent Instructions

In [4]:
# Data Agent specifications
AgentName = "DA_TestAutomationRun"

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 12, Finished, Available, Finished, False)

In [5]:
# create DataAgent
data_agent = create_data_agent(AgentName)

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 13, Finished, Available, Finished, False)

/nfs4/pyenv-0cccd8be-298f-471d-be04-f7ab6b5e17de/lib/python3.11/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)


In [6]:
# Specify the Data Agent instructions, below instructions are in markdown language
data_agent.update_configuration(
    instructions="""
# General setup
## Purpose
Answer business questions using the connected enterprise data sources.  
Provide clear, concise, and accurate insights based only on available data.

## Accuracy Rules
- Only use information from connected data sources.
- Do not fabricate data.
- Ask a clarifying question if the request is ambiguous.

## Response Style
Responses should be:
- Clear and concise
- Written in professional business language
- Focused on the direct answer first
- Structured when useful (short summary + optional table)

## Planning Approach
When answering a question:
1. Understand the user intent and identify the required metric, dimension, and time period.
2. Select the most appropriate data source based on the type of question and data completeness.
3. Use existing business measures where available.
4. Validate that the result logically matches the question.
5. Return the answer with a short explanation if needed.

## Conflict Resolution
If multiple data sources could answer the question, choose the source that contains the **most complete data for the requested time period**.

---

# Case specifics
## Terminology
Use consistent business terminology:
- **Retail price** → The average price for the fuel on the given day.
- **Fuel types** → BenzineEuro95 (gasoline), Diesel, and LPG (gas). BenzineEuro95 might also be known as "Euro 95" or "EURO 95" or any other way of spelling it, with or without spaces, uppercase and lowercase. 

Use business-friendly language rather than technical table or column names.

---

## Business and calculation rules
- The advised retail price stored in the **Semantic Model** is only available on dates where a **fuel transaction occurred**.
- When analyzing **price trends, price increases, or prices on dates without transactions**, use the **Lakehouse dataset**, because it contains the complete historical timeline of advised fuel prices.
- All fuel values are recorded in **liters**.  
  Convert to gallons by dividing by **0.264172**.

---

## Trend Analysis
When a question asks for:
- **price trends**
- **price increases or decreases**
- **historical price comparisons across years**

---

## Time Interpretation
If no time period is specified, assume the **most recent complete period**.

Common interpretations:
- *This year* → Current calendar year  
- *Last year* → Previous calendar year  
- *This month* → Current calendar month  
- *Last month* → Previous calendar month
"""
)

config = data_agent.get_configuration()
print(config) 
# Note that the output is one long text string, however in the Data Agent config it is correctly formatted. 

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 14, Finished, Available, Finished, False)

DataAgentConfiguration(instructions='\n# General setup\n## Purpose\nAnswer business questions using the connected enterprise data sources.  \nProvide clear, concise, and accurate insights based only on available data.\n\n## Accuracy Rules\n- Only use information from connected data sources.\n- Do not fabricate data.\n- Ask a clarifying question if the request is ambiguous.\n\n## Response Style\nResponses should be:\n- Clear and concise\n- Written in professional business language\n- Focused on the direct answer first\n- Structured when useful (short summary + optional table)\n\n## Planning Approach\nWhen answering a question:\n1. Understand the user intent and identify the required metric, dimension, and time period.\n2. Select the most appropriate data source based on the type of question and data completeness.\n3. Use existing business measures where available.\n4. Validate that the result logically matches the question.\n5. Return the answer with a short explanation if needed.\n\n##

## Data sources
This section covers various tasks:
- Add a data source to the newly created data agent
- Add data source instructions for the specified data source
- Select tables from the data source for the data agent to reason over

In [9]:
# add a data source
lakehouse_name = "LH_Store_Gold" 
data_agent.add_datasource(lakehouse_name, type="lakehouse") 

# datasource type could be: lakehouse, kqldatabase, warehouse or semanticmodel

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 17, Finished, Available, Finished, False)

Datasource(567587fd-eba0-41f6-b782-4db3991feaf6)

In [10]:
# Verify added data source(s) and selected table configs
datasource = data_agent.get_datasources()[0] # only returns the first data source, customize where needed
datasource.pretty_print()

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 18, Finished, Available, Finished, False)

 Schemas
  | dbo
  |  | Tables
  |  |  | date
  |  |  |  | DateKey
  |  |  |  | Year
  |  |  |  | Month
  |  |  |  | Day
  |  |  |  | DayOfWeek
  |  |  |  | DayName
  |  |  |  | Quarter
  |  |  |  | MonthName
  |  |  |  | YearMonth
  |  |  |  | YearQuarter
  |  |  | car
  |  |  |  | CarKey
  |  |  |  | LicensePlate
  |  |  |  | Brand
  |  |  |  | Type
  |  |  |  | StartDate
  |  |  |  | EndDate
  |  |  | transactions
  |  |  |  | CarKey
  |  |  |  | Date
  |  |  |  | FuelType
  |  |  |  | Liters
  |  |  |  | Price
  |  |  |  | Mileage
  |  |  |  | AverageRetailPricePerLiter
  |  |  |  | PaidPricePerLiter
  |  |  |  | PriceDifferencePerLiter
  |  |  |  | TotalPriceDifference
  |  |  | recommended retail price
  |  |  |  | ID
  |  |  |  | Date
  |  |  |  | BenzineEuro95
  |  |  |  | Diesel
  |  |  |  | LPG


'LH_Store_Gold'

In [11]:
# Add tables to data agent
datasource.select("Schemas", "dbo", "Tables", "date")
datasource.select("Schemas", "dbo", "Tables", "recommended retail price")

# Note that the order in above select statement should be an exact match with the pattern that was returned in the previous step

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 19, Finished, Available, Finished, False)

In [12]:
# Update data source instructions
ds_notes = """
## Lakehouse – Historical Advised Fuel Prices
The lakehouse contains a **complete historical dataset of advised retail fuel prices**.

Characteristics:
- Covers **all dates from 1 January 2006 until today**.
- Updated **weekly**.
- Contains the advised price for three fuel types.

Available columns:
- Date
- BenzineEuro95
- Diesel
- LPG

Typical use cases:
- Analyzing fuel price trends over time.
- Determining price increases or decreases between two dates.
- Retrieving advised prices for dates without transactions.
"""
datasource.update_configuration(instructions=ds_notes)
datasource.get_configuration()["additional_instructions"]

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 20, Finished, Available, Finished, False)

'\n## Lakehouse – Historical Advised Fuel Prices\nThe lakehouse contains a **complete historical dataset of advised retail fuel prices**.\n\nCharacteristics:\n- Covers **all dates from 1 January 2006 until today**.\n- Updated **weekly**.\n- Contains the advised price for three fuel types.\n\nAvailable columns:\n- Date\n- BenzineEuro95\n- Diesel\n- LPG\n\nTypical use cases:\n- Analyzing fuel price trends over time.\n- Determining price increases or decreases between two dates.\n- Retrieving advised prices for dates without transactions.\n'

## Publish Data Agent

In [16]:
data_agent.publish(AgentName)
# Note that it can take a little moment before the published version is available

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 24, Finished, Available, Finished, False)

In [17]:
data_agent.get_configuration()

StatementMeta(, b92880ec-7479-4857-bc31-a499a6f88797, 25, Finished, Available, Finished, False)

DataAgentConfiguration(instructions='\n# General setup\n## Purpose\nAnswer business questions using the connected enterprise data sources.  \nProvide clear, concise, and accurate insights based only on available data.\n\n## Accuracy Rules\n- Only use information from connected data sources.\n- Do not fabricate data.\n- Ask a clarifying question if the request is ambiguous.\n\n## Response Style\nResponses should be:\n- Clear and concise\n- Written in professional business language\n- Focused on the direct answer first\n- Structured when useful (short summary + optional table)\n\n## Planning Approach\nWhen answering a question:\n1. Understand the user intent and identify the required metric, dimension, and time period.\n2. Select the most appropriate data source based on the type of question and data completeness.\n3. Use existing business measures where available.\n4. Validate that the result logically matches the question.\n5. Return the answer with a short explanation if needed.\n\n##